<a href="https://colab.research.google.com/github/kingdragonlord/sequential-deep-learning-architectures/blob/main/sequential_deep_learning_architectures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INSTRUCTIONS:

For each of the following 10 tasks:


1. Pick a high level architectural paradigm (e.g. sequence labeling, language modeling, sequence classification, encoder-decoder) and explain your choice.


2. Give me the blueprints of your model, that is, describe the components of your model and how they fit together to achieve the overall task: LSTM vs RNN cells vs. something else? Stacked RNNs? Bidirectional RNNs? Use of other layers or blocks? Justify each choice. Note, you don’t need to actually create the model in code, but give me at least enough detail to where I would be able to reconstruct your model in code exactly as you envision it.


3. Describe, in as much detail as possible, how you would form the dataset for this task. For tasks with an asterisk next to them, create a small example dataset in Colab. Remember that each sequence in a batch of data needs to have the same length, but different batches may have differing lengths (that is, maximum sequence lengths). You may find the following helpful:
`pad_sequence` from `torch.nn.utils.rnn`
`pad_packed_sequence` from `torch.nn.utils.rnn`
`pack_padded_sequence` from `torch.nn.utils.rnn`


4. Describe the shapes of data as they pass through your network, from inputs to outputs.


5. Describe the shapes and nature of the outputs of your model and how you would use them to make decisions.


**NOTE: For a lot of the models I choose the Transformer based architecture since it is the most modern architecture, but a lot of the time a RNN with LSTM layers works, specially for smaller models or models with smaller datasets.**

# TASK 1 - Text Generation
 Text Generation (*) – Train a model to generate text in the style of Shakespeare, news articles, or programming code.

1. The high level architectural paradigm that I would use to train a model in the style of Shakespeare is language modeling.Language modeling involves predicting the next word in a sequence given the context of the previous words.

2. For the model blueprint I would use a unidirectional LSTM with an embedding layer and a few stacked LSTM layers. The embedding layer will capture the semantic meaning of the words, and the LSTM layers are capable of capturing long range dependencies in sequential data such as Shakespeare's writing style. I must also include a dropout layer to prevent overfitting and a final linear and SoftMax layer to get out the next probable possible word in the sequence.

3. To form the dataset for this task, I can use a large corpus of data from Shakespeare works. Then, I would split the text into chunks and predict the next word in the sequence (Sliding window with a certain context size). To handle varying lengths I will use the pad_sequence function from torch.nn.utils.rnn to pad the input to the maximum length in the batch, ensuring all batches have the same length.

In [ ]:
import torch
from collections import Counter
from torch.nn.utils.rnn import pad_sequence

# Sample text data
text_data = [
    "To be or not to be that is the question",
    "Whether tis nobler to suffer the slings and arrows",
    "Of outrageous fortune or to take arms against"
]

# Tokenize text data
tokens = [sentence.split() for sentence in text_data]

# 1. Build vocabulary using collections.Counter
token_counts = Counter(token for sentence in tokens for token in sentence)
special_tokens = ["<pad>", "<unk>"]
token_to_idx = {tok: idx for idx, tok in enumerate(special_tokens)}
for token, _ in token_counts.most_common():
    if token not in token_to_idx:
        token_to_idx[token] = len(token_to_idx)

# Flatten corpus to a single continuous sequence of token IDs
flat_tokens = [token_to_idx[token] for sentence in tokens for token in sentence]

# 2. Sliding window dataset construction (Raschka Ch. 2.6)
# Each input x and target y have length max_length, shifted by 1 token
max_length = 4
stride = 1

input_sequences = []
target_sequences = []

for i in range(0, len(flat_tokens) - max_length, stride):
    x_chunk = flat_tokens[i : i + max_length]
    y_chunk = flat_tokens[i + 1 : i + max_length + 1]
    input_sequences.append(torch.tensor(x_chunk))
    target_sequences.append(torch.tensor(y_chunk))

# Batch stacking
x_batch = torch.stack(input_sequences)
y_batch = torch.stack(target_sequences)

print("Input sequences (X) shape:", x_batch.shape)   # (batch_size, max_length)
print("Target sequences (Y) shape:", y_batch.shape)  # (batch_size, max_length)

Input sequences (X) shape: torch.Size([23, 4])
Target sequences (Y) shape: torch.Size([23, 4])


4. Lets talk about the shapes as the pass through the network:
Input: (batch_size, sequence_length)

Embedding Layer: (batch_size, sequence_length, embedding_dim)

LSTM Layers: (batch_size, sequence_length, hidden_dim)

Linear Layer / Logits: (batch_size, sequence_length, vocab_size)

Target: (batch_size, sequence_length) — cross-entropy loss is computed across all time-steps by flattening the predictions to (batch_size * sequence_length, vocab_size) against target (batch_size * sequence_length).

5. Nature of the outputs and decision making:
The output of the model is a probability distribution over the possible next tokens in the sequence. To generate text, we can sample from this distribution to select the next token.
  
  To make decisions we can use the output probability distribution to:
Greedy sampling, Top-K sampling, or temperature sampling.
Then the model can be evaluated using metrics such as perplexity.

# TASK 2 - Machine Translation

Machine Translation (*) – Implement a model for translating between languages (e.g., English to German).

1. The high-level architectural paradigm for the task of machine translation is Encoder-Decoder. The encoder-decoder paradigm is well-suited for machine translation tasks.

2. For machine translation, I propose the following architecture:
Embedding Layers- Both the encoder and decoder will use embedding layers to convert information into dense vector representations.
Encoder - Preferably a Transformer model that takes in the input sequence and outputs a continuous representation of the input sequence.
Decoder - Transformer model that takes in the output of the encoder and generates the translated output sequence.
Attention Mechanism: To help the decoder focus on different parts of the input sequence when generating the output sequence.
The reason for choosing Transformer as the encoder and decoder is due to their ability to capture complex patterns and relationships in the input sequence.

3. To form the dataset, I will use a parallel corpus of paired source and target sentences (e.g., WMT English–German). Rather than relying on whitespace splitting—which breaks down on languages without explicit word boundaries (e.g., Chinese, Japanese) and struggles with German compound words (e.g., Donaudampfschifffahrt)—the text will be preprocessed using subword tokenization such as Byte-Pair Encoding (BPE) or SentencePiece (e.g., using Hugging Face tokenizers or sentencepiece). Subwords prevent out-of-vocabulary (OOV) tokens and naturally decompose long German compounds into manageable constituent subwords. Special tokens (`<bos>`, `<eos>`, `<pad>`, `<unk>`) will be prepended and appended to mark sequence boundaries. Batches are padded to the maximum sequence length in each batch using torch.nn.utils.rnn.pad_sequence.

In [ ]:
import torch
from collections import Counter
from torch.nn.utils.rnn import pad_sequence

# Sample parallel text data
english_data = [
    "Hello how are you",
    "What is your name",
    "I am from the United States"
]
german_data = [
    "Hallo wie geht es dir",
    "Wie heißt du",
    "Ich komme aus den Vereinigten Staaten"
]

# Note: In production, use subword tokenization (BPE/SentencePiece) to handle
# German compound words and continuous scripts without whitespaces.
english_tokens = [sentence.split() for sentence in english_data]
german_tokens = [sentence.split() for sentence in german_data]

# Build vocabularies using collections.Counter
def build_vocab(tokenized_sentences, special_tokens=["<pad>", "<unk>", "<bos>", "<eos>"]):
    counts = Counter(token for sent in tokenized_sentences for token in sent)
    vocab = {tok: idx for idx, tok in enumerate(special_tokens)}
    for token, _ in counts.most_common():
        if token not in vocab:
            vocab[token] = len(vocab)
    return vocab

english_token_to_idx = build_vocab(english_tokens)
german_token_to_idx = build_vocab(german_tokens)

# Convert tokens to indices with boundary tokens
english_indexed = [
    torch.tensor([english_token_to_idx["<bos>"]] + [english_token_to_idx[t] for t in s] + [english_token_to_idx["<eos>"]])
    for s in english_tokens
]
german_indexed = [
    torch.tensor([german_token_to_idx["<bos>"]] + [german_token_to_idx[t] for t in s] + [german_token_to_idx["<eos>"]])
    for s in german_tokens
]

padded_english_inputs = pad_sequence(english_indexed, batch_first=True, padding_value=english_token_to_idx["<pad>"])
padded_german_targets = pad_sequence(german_indexed, batch_first=True, padding_value=german_token_to_idx["<pad>"])

print("English Batch Shape:", padded_english_inputs.shape)
print("German Batch Shape:", padded_german_targets.shape)

English Batch Shape: torch.Size([3, 8])
German Batch Shape: torch.Size([3, 8])


4. Shapes as they pass through the network:
Input:The input to the model is a tensor of shape (batch_size, sequence_length)
Encoder: The output of the encoder is a tensor of shape (batch_size, sequence_length, hidden_dim)
Decoder: The output of the decoder is a tensor of shape (batch_size, sequence_length, vocab_size)
Attention Mechanism: The attention mechanism will output a tensor of shape (batch_size, sequence_length, sequence_length) representing the attention weights.

5. The output of the model is a probability distribution over the possible next tokens in the target sequence. To make decisions we can use:
Greedy decoding, Top-K Sampling, and Beam Search.
We can use metrics such as BLUE/ROUGE scores to evaluate the model.

# TASK 3 - Speech Recognition

Speech Recognition – Use a model to transcribe speech into text (e.g., building an Automatic Speech Recognition (ASR) system).
Hint: For feature extraction from audio signals, you can use MFCC features. I put a demo together for you here. You don’t need to know the details of how MFCC works for the sake of this class, but it is a feature extraction technique that converts chunks of audio into fixed length vectors. So, analogous to our tokenization process for text, you can imagine that we split our audio signal into small overlapping chunks, then apply MFCC to each chunk to get a sequence of fixed length vectors, in our familiar form (B, T, D). Once you have a sequence of fixed length vectors, you can use RNNs (or other models) to process them.

1. The high-level architectural paradigm for the task of speech recognition is Sequence-to-Sequence. The sequence-to-sequence paradigm is well-suited for speech recognition tasks as it allows the model to learn a mapping between the input audio sequence and the output text sequence.

2. This is the architecture I propose for the model:
Feature extraction: The input audio signal will be processed using MFCC (Mel-Frequency Cepstral Coefficients) feature extraction to obtain a sequence of fixed-length vectors. This will be done by splitting the audio signal into small overlapping chunks and applying MFCC to each chunk.

  Encoder: The extracted MFCC features will be fed into an encoder stacked Transformer model to capture patterns and structure in the audio sequence.

  Decoder: The output of the encoder will be passed through a decoder, which will be an Stacked Transformer.The decoder will generate the output text sequence based on the encoded representation of the input audio sequence.

  Attention Mechanism: To help the decoder focus on different parts of the input audio sequence when generating the output text sequence, an attention mechanism will be used.

  The choice of using a Transformer as the encoder is due to their ability to capture complex patterns and relationships in the input audio sequence.

3. To form the dataset for this task, I can use a large corpus of audio-text pairs. The audio data will be preprocessed by extracting MFCC features from the audio signals.The dataset will be created by pairing the extracted MFCC features with the corresponding text transcripts. The input to the model will be a sequence of MFCC features, and the target output will be a sequence of tokens representing the text transcript. To handle sequences of varying lengths, I will use pad_sequence.

4. Shapes of the data as they pass through the network:
Input: The input to the model is a tensor of shape (batch_size, sequence_length, feature_dim), where batch_size is the number of sequences in a batch, sequence_length is the length of each sequence, and feature_dim is the dimensionality of the MFCC features.
Encoder: The output of the encoder is a tensor of shape (batch_size, sequence_length, hidden_dim)
Decoder: The output of the decoder is a tensor of shape (batch_size, sequence_length, vocab_size)
Attention Mechanism: The attention mechanism will output a tensor of shape (batch_size, sequence_length, sequence_length)

5. Nature of outputs and decision making:
The output of the model is a probability distribution over the possible next tokens in the output text sequence. To make decisions we can use:
Greedy decoding, Beam search and we can evaluate the model using metrics such as Word Error Rate (WER) or Character Error Rate (CER)

In [ ]:
# CODE FROM DEMO
import torchaudio
import torchaudio.transforms as transforms
import torch
import matplotlib.pyplot as plt
from torch.nn.utils.rnn import pad_sequence
import os
import urllib.request

# MFCC feature extraction for audio files
def extract_mfcc(filepath):
  waveform, sample_rate = torchaudio.load(filepath)
  if waveform.shape[0] > 1: # convert audio to mono
    waveform = waveform.mean(dim=0, keepdim=True)
  waveform = torchaudio.functional.resample(waveform, orig_freq=sample_rate, new_freq=16000)
  mfcc = mfcc_transform(waveform)
  return mfcc.squeeze(0).T

base_url = "https://download.pytorch.org/torchaudio/tutorial-assets/"
file_names = [
    "steam-train-whistle-daniel_simon.wav",
    "Lab41-SRI-VOiCES-src-sp0307-ch127535-sg0042.wav"
]

audio_files = []
for name in file_names:
    if not os.path.exists(name):
        urllib.request.urlretrieve(base_url + name, name)
    audio_files.append(name)

mfcc_transform = transforms.MFCC(
    sample_rate=16000,
    n_mfcc=13,
    melkwargs={"n_mels": 40, "n_fft": 400, "hop_length": 160}
)

# Feature extraction for batch of audio files
batch_mfcc = [extract_mfcc(f) for f in audio_files]

# Pad sequences to match longest sequence in batch
batch_padded = pad_sequence(batch_mfcc, batch_first=True, padding_value=0.0)  # Shape: (batch_size, max_T, n_mfcc)

print("Batch MFCC Shape:", batch_padded.shape)  # (B, T, D)

Batch MFCC Shape: torch.Size([2, 341, 13])


# TASK 4 - Named Entity Recognition

Named Entity Recognition (*) – Implement a system to extract entities like names, dates, and organizations from text.

1. The high-level architectural paradigm for the task of named entity recognition is Sequence Labeling. The sequence labeling paradigm is well-suited for named entity recognition tasks as it allows the model to learn to predict a label for each token in the input sequence.

2. For the model architecture I propose an encoder only transformer with the following components:
Embedding layer, Encoder, Linear layer, SoftMax. The choice of using a Transformer as the encoder is due to their ability to capture complex patterns and relationships in the input sequence.

3. To form the dataset for this task, I will use a large corpus of labeled text data (e.g., text with named entities annotated). The text data will be preprocessed by tokenizing the text into individual words or subwords.The dataset will be created by pairing the tokenized text with the corresponding labels. The input to the model will be a sequence of tokens, and the target output will be a sequence of labels representing the named entities.To handle sequences of varying lengths, I will use the pad_sequence function.

In [ ]:
import torch
from collections import Counter
from torch.nn.utils.rnn import pad_sequence

text_data = [
    ["John", "is", "going", "to", "New", "York"],
    ["Jane", "lives", "in", "London"]
]
labels = [
    ["B-PER", "O", "O", "O", "B-LOC", "I-LOC"],
    ["B-PER", "O", "O", "B-LOC"]
]

# Build token and label vocabularies with Counter
token_counts = Counter(tok for sent in text_data for tok in sent)
token_to_idx = {"<pad>": 0, "<unk>": 1}
for tok, _ in token_counts.most_common():
    if tok not in token_to_idx:
        token_to_idx[tok] = len(token_to_idx)

label_counts = Counter(lbl for sent in labels for lbl in sent)
label_to_idx = {"<pad>": 0}
for lbl, _ in label_counts.most_common():
    if lbl not in label_to_idx:
        label_to_idx[lbl] = len(label_to_idx)

indexed_tokens = [torch.tensor([token_to_idx.get(t, 1) for t in sent]) for sent in text_data]
indexed_labels = [torch.tensor([label_to_idx[l] for l in sent]) for sent in labels]

padded_inputs = pad_sequence(indexed_tokens, batch_first=True, padding_value=0)
padded_targets = pad_sequence(indexed_labels, batch_first=True, padding_value=0)

print(padded_inputs.shape)
print(padded_targets.shape)

torch.Size([2, 6])
torch.Size([2, 6])


4. Shapes of the data as they pass through the network:
Input: The input to the model is a tensor of shape (batch_size, sequence_length)
Embedding Layer: The output of the embedding layer is a tensor of shape (batch_size, sequence_length, embedding_dim)
Encoder: The output of the encoder is a tensor of shape (batch_size, sequence_length, hidden_dim),
Linear Layer and Softmax Output: The final output of the model is a tensor of shape (batch_size, sequence_length, num_labels)

5.  The output of the model is a probability distribution over the possible labels for each token. To make predictions, we can select the label with the highest probability for each token.
We can use the BIO (Beginning-Inside-Outside) tagging scheme to identify the start and end of named entities.The performance of the model can be evaluated using metrics such as precision, recall, and F1-score.

# TASK 5 - Toxic Comment Classification

Toxic Comment Classification (*) - Create a model to classify whether a comment on an online platform is toxic, harmful, or abusive.

1. The high-level architectural paradigm for the task of toxic comment classification is Sequence Classification. The sequence classification paradigm is well-suited for toxic comment classification tasks as it allows the model to learn to predict a label (toxic or not toxic) for the entire input sequence.

2. I propose an Encoder Transformer based model with an:
Embedding layer to get a dense word representation
Encoder transformer model to capture patterns and structure.
Pooling layer to obtain a fixed size representation of the input sequence
Linear layer and a sigmoid activation function to produce the probability score of weather a comment is toxic or not.

3. To form the dataset for this task, I will use a large corpus of labeled text data (e.g., comments with toxicity labels). The text data will be preprocessed by tokenizing the text into individual words or subwords. The dataset will be created by pairing the tokenized text with the corresponding labels. The input to the model will be a sequence of tokens, and the target output will be a binary label (toxic or not toxic). To handle sequences of varying lengths, I will use the pad_sequence function.

In [ ]:
import torch
from collections import Counter
from torch.nn.utils.rnn import pad_sequence

text_data = [
    ["This", "is", "a", "toxic", "comment"],
    ["This", "is", "a", "normal", "comment"]
]
labels = torch.tensor([1, 0], dtype=torch.float32)

# Build vocabulary with Counter
token_counts = Counter(tok for sent in text_data for tok in sent)
token_to_idx = {"<pad>": 0, "<unk>": 1}
for tok, _ in token_counts.most_common():
    if tok not in token_to_idx:
        token_to_idx[tok] = len(token_to_idx)

indexed_tokens = [torch.tensor([token_to_idx[t] for t in sent]) for sent in text_data]
padded_inputs = pad_sequence(indexed_tokens, batch_first=True, padding_value=0)

print(padded_inputs.shape)

torch.Size([2, 5])


4. Shapes of data as they pass through the network:
Input: The input to the model is a tensor of shape (batch_size, sequence_length)
Embedding Layer: The output of the embedding layer is a tensor of shape (batch_size, sequence_length, embedding_dim)
Encoder: The output of the encoder is a tensor of shape (batch_size, sequence_length, hidden_dim)
Pooling Layer: The output of the pooling layer is a tensor of shape (batch_size, hidden_dim).
Linear Layer and Sigmoid Output: The final output of the model is a tensor of shape (batch_size, 1)

5. The output of the model is a probability score indicating whether the comment is toxic or not. To make predictions, we can threshold the probability score (e.g., 0.5) to determine whether the comment is toxic or not.The performance of the model can be evaluated using metrics such as accuracy, precision, recall, and F1-score.

# TASK 6 - Grammar Correction

Grammar Correction – Develop a model that predicts and corrects grammatical errors in text (not a spell checker, but something that can suggest rewrites of sentences/passages).

1. The high-level architectural paradigm for the task of grammar correction is Sequence-to-Sequence. The sequence-to-sequence paradigm is well-suited for grammar correction tasks as it allows the model to learn a mapping between the input sequence ( potentially grammatically incorrect text) and the output sequence (corrected text).

2. I propose an encoder - decoder transformer model for the architecture with the following components:
Embedding layers on encoder and decoder : For dense vector representations
Encoder: The input to the model will be a sequence of tokens
Decoder: The output of the encoder will be passed through a decoder
Attention Mechanism: The decoder will use an attention mechanism to focus on different parts of the input sequence when generating the output sequence.
Transformers have a great ability to capture complex patterns and relationships in the input sequence.

3. I will use a large corpus of text data with grammatical errors and their corresponding corrections. The text data will be preprocessed by tokenizing the text into individual words or subwords.The dataset will be created by pairing the tokenized text with grammatical errors with the corresponding corrected text. The input to the model will be a sequence of tokens with grammatical errors, and the target output will be a sequence of tokens representing the corrected text. To handle sequences of varying lengths, I will use the pad_sequence function.

4. Shapes of data as it passes through the network:
Input: The input to the model is a tensor of shape (batch_size, sequence_length)
Encoder: The output of the encoder is a tensor of shape (batch_size, sequence_length, hidden_dim)
Decoder: The output of the decoder is a tensor of shape (batch_size, sequence_length, vocab_size)
Attention Mechanism: The attention mechanism will output a tensor of shape (batch_size, sequence_length, sequence_length)

5. Nature of the outputs and decision making:
The output of the model is a probability distribution over the possible next tokens in the output sequence. To generate the corrected text, we can sample from this distribution to select the next token. To make decisions we can use the output distribution to do:
Greedy decoding, or Beam search and to evaluate the mode we can use BLEU score, ROUGE score, or Grammatical Error Rate (GER).

# TASK 7 - Voice Cloning or Speech Synthesis

Voice Cloning or Speech Synthesis – Create a model to generate realistic-sounding speech from text.

1. The high-level architectural paradigm for the task of voice cloning or speech synthesis is Text-to-Speech (TTS). The TTS paradigm involves generating speech from text input.

2. I propose a model architecture with the following components:
Text Encoder: The input to the model will be a sequence of text, which will be fed into a text encoder, a Transformer model.
Vocoder: The output of the text encoder will be passed through a vocoder, which will generate the speech waveform. The vocoder can be a WaveNet or HiFi-GAN model.
Mel-Spectrogram Prediction: Before passing the output to the vocoder, the model will predict the Mel-spectrogram of the speech. The Mel-spectrogram is a representation of the speech signal in the frequency domain.

3. To form the dataset for this task, I will use a large corpus of text-speech pairs (e.g., text with corresponding audio recordings). The text data will be preprocessed by tokenizing the text into individual words or subwords. The dataset will be created by pairing the tokenized text with the corresponding audio recordings. The input to the model will be a sequence of tokens, and the target output will be the Mel-spectrogram or speech waveform. To handle sequences of varying lengths, I will use padding and masking techniques to ensure that the input and output sequences are properly aligned.

4. Shapes of the data as it passes through the network:
Input: The input to the model is a tensor of shape (batch_size, sequence_length)
Text Encoder: The output of the text encoder is a tensor of shape (batch_size, sequence_length, hidden_dim)
Mel-Spectrogram Prediction: The output of the Mel-spectrogram prediction module is a tensor of shape (batch_size, mel_bins, sequence_length)
Vocoder: The output of the vocoder is a tensor of shape (batch_size, waveform_length),

5. Nature of the outputs and decision making:
The output of the model is a speech waveform that corresponds to the input text. To evaluate the quality of the generated speech, we can use metrics such as Mean Opinion Score (MOS), Short-Time Objective Intelligibility (STOI), or Perceptual Evaluation of Speech Quality (PESQ).

# TASK 8 - Image Captioning

Image Captioning – Create a model to generate captions for given images.

1. The high-level architectural paradigm for the task of image captioning is Encoder-Decoder. The encoder-decoder paradigm is well-suited for image captioning tasks as it allows the model to learn a mapping between the input image and the output caption.

2. For image captioning I propose a model architecture with the following components:
Image Encoder: The input to the model will be an image, which will be fed into an image encoder, such as a Convolutional Neural Network (CNN). The image encoder will learn to extract features from the input image.
Caption Decoder: The output of the image encoder will be passed through a caption decoder, which will be a Transformer model. The caption decoder will generate the output caption based on the encoded representation of the input image.
The attention mechanism: To help the caption decoder focus on different parts of the input image when generating the output caption

  The choice of using a CNN as the image encoder is due to their ability to extract features from images. The use of an Transformer as the caption decoder is due to their ability to capture sequential dependencies in the output caption.

3. To form the dataset for this task, I will use a large corpus of image-caption pairs (e.g., images with corresponding captions). The image data will be preprocessed by resizing and normalizing the images. The dataset will be created by pairing the preprocessed images with the corresponding captions. The input to the model will be an image, and the target output will be a sequence of tokens representing the caption. To handle sequences of varying lengths, I can use padding and masking techniques such as pack_padded_sequence, pad_sequence.

4. Input: The input to the model is an image of shape (batch_size, channels, height, width), where batch_size is the number of images in a batch, channels is the number of color channels, and height and width are the dimensions of the image.
Image Encoder: The output of the image encoder is a tensor of shape (batch_size, hidden_dim)
Caption Decoder: The output of the caption decoder is a tensor of shape (batch_size, sequence_length, vocab_size)
Attention Mechanism: The attention mechanism will output a tensor of shape (batch_size, sequence_length, hidden_dim)

5. The output of the model is a probability distribution over the possible next tokens in the caption. To generate the caption, we can sample from this distribution to select the next token. To make decisions we can use the output probability to do :
Greedy decoding, or Beam search and we can evaluate the model using BLEU, ROUGE, or CIDEr scores.

# TASK 9 - Human Activity Recognition

Human Activity Recognition – Create a model to classify human actions (e.g., walking, running, jumping) from sensor data or video sequences.

1. The high-level architectural paradigm for the task of human activity recognition is Sequence Classification. The sequence classification paradigm is well-suited for human activity recognition tasks as it allows the model to learn to predict a label (activity) for the entire input sequence.

2. To be able to complete the task of human activity recognition, I will need a model architecture with the following components:
Sensor Data or Video Preprocessing: The input to the model will be preprocessed sensor data or video sequences. The preprocessing step may involve data normalization, feature extraction, or other techniques to prepare the data for modeling.
Feature Extraction: The preprocessed data will be fed into a feature extraction module, which will extract relevant features from the data. For sensor data, this may involve using techniques such as Fast Fourier Transform (FFT) or Continuous Wavelet Transform (CWT). For video sequences, this may involve using Convolutional Neural Networks (CNNs) to extract spatial and temporal features.
Sequence Modeling: The extracted features will be fed into a sequence modeling module, which will be a Transformer model. The sequence modeling module will learn to capture the patterns and structures in the input sequence.
Classification: The output of the sequence modeling module will be passed through a classification module, which will predict the activity label.

3. To form the dataset, I will use a large corpus of labeled sensor data or video sequences. The data will be preprocessed and split into training, validation, and testing sets. The dataset will be created by pairing the preprocessed data with the corresponding activity labels. The input to the model will be a sequence of data, and the target output will be an activity label. To handle sequences of varying lengths, I will use padding and masking techniques such as pack_padded_sequence, pad_sequence.

4. Shapes of data as it passes through the network:
Input: The input to the model is a tensor of shape (batch_size, sequence_length, feature_dim) where feature_dim is the dimensionality of the feature space.
Feature Extraction: The output of the feature extraction module is a tensor of shape (batch_size, sequence_length, feature_dim) where feature_dim is the dimension of extracted features.
Sequence Modeling: The output of the sequence modeling module is a tensor of shape (batch_size, hidden_dim)
Classification: The output of the classification module is a tensor of shape (batch_size, num_classes)

5. Nature of the outputs and decision making:
The output of the model is a probability distribution over the possible activity classes. To make predictions, we can select the class with the highest probability.The performance of the model can be evaluated using metrics such as accuracy, precision, recall, and F1-score.

# TASK 10 - Speech Emotion Recognition

Speech Emotion Recognition – Create a model to classify emotions from speech audio.

1. The high-level architectural paradigm for the task of speech emotion recognition is Sequence Classification.

2. For the task of speech emotion recognition, I suggest a model with the following architecture:
Audio Preprocessing: The input to the model will be preprocessed audio data, which may involve techniques such as Mel-Frequency Cepstral Coefficients (MFCC) extraction or spectrogram computation.
Feature Extraction: The preprocessed audio data will be fed into a feature extraction module, which will extract relevant features from the data. This may involve using techniques such as Convolutional Neural Networks (CNNs) or Recurrent Neural Networks (RNNs).
Sequence Modeling: The extracted features will be fed into a sequence modeling module, which will be an RNN or Transformer model. The sequence modeling module will learn to capture the patterns and structures in the input sequence.
Classification: The output of the sequence modeling module will be passed through a classification module, which will predict the emotion label.

3. To form the dataset for this task, I will use a large corpus of labeled audio data. The data will be preprocessed and split into training, validation, and testing sets. The dataset will be created by pairing the preprocessed audio data with the corresponding emotion labels. The input to the model will be a sequence of audio data, and the target output will be an emotion label. To handle sequences of varying lengths, I will use padding and masking techniques mentioned earlier such as pad_sequence and pack_padded_sequence.

4. Shapes of data as it passes through the network:
Input: The input to the model is a tensor of shape (batch_size, sequence_length, feature_dim)
Feature Extraction: The output of the feature extraction module is a tensor of shape (batch_size, sequence_length, feature_dim)
Sequence Modeling: The output of the sequence modeling module is a tensor of shape (batch_size, hidden_dim)
Classification: The output of the classification module is a tensor of shape (batch_size, num_classes)

5. Nature of the outputs and decision making:The output of the model is a probability distribution over the possible emotion classes. To make predictions, we can select the class with the highest probability. The performance of the model can be evaluated using metrics such as accuracy, precision, recall, and F1-score.